# Grafo Verb <-> Verb (Hiperlink)

In [ ]:
# CÉLULA 1: CONFIGURAÇÃO E FUNÇÕES AUXILIARES
# Execute esta célula uma vez para carregar as configurações e funções na memória.

import json
import pandas as pd
import networkx as nx
import community as community_louvain
import random
from pathlib import Path
from typing import List, Dict, Any, Optional
import math
import pickle
import colorsys
import time

# --- CONFIGURAÇÃO  ---
INPUT_DIR = Path('../dados/dados_com_flags_redirecionamento') 
# Ficheiro de entrada (resultado do script de coleta)
INPUT_FILENAME_WITH_FLAGS = 'dados_api_com_flags_e_refs.json'
REDIRECT_MAP_FILENAME = 'redirect_map.json'
# Ficheiros de saída intermediários e finais
OUTPUT_NORMALIZED = 'dados_com_referencias_normalizadas.json' 
METRICS_OUTPUT_FILENAME = 'dados_com_metricas_v2_novo.json'
POSITIONS_OUTPUT_FILENAME = 'dados_com_posicoes_v2_novo.json'
FINAL_REFINED_OUTPUT_FILENAME = 'dados_com_posicoes_v3_novo.json'
SHORTEST_PATHS_FILENAME = 'shortest_paths_dist.pkl' # Ficheiro para as distâncias

# Parâmetros dos algoritmos
LOUVAIN_RESOLUTION = 0.9
RANDOM_SEED = 42
LAYOUT_SCALE_FACTOR = 8000

print("✅ Célula 1: Configurações e funções carregadas.")

# --- FUNÇÕES AUXILIARES ---

def carregar_dados_json(filepath: Path) -> Optional[List[Dict[str, Any]]]:
    """
    Carrega dados de um arquivo JSON e retorna especificamente a lista
    armazenada sob a chave 'verbetes_completo'.
    """
    if not filepath.exists():
        print(f"ERRO: Arquivo não encontrado em '{filepath}'")
        return None # Retorna None se o arquivo não existe

    print(f"Carregando dados de '{filepath}'...")
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            # Carrega o JSON inteiro (que é um dicionário)
            dados_completos = json.load(f)

        lista_verbetes = dados_completos.get('verbetes_completo', [])

        if not isinstance(lista_verbetes, list):
             print(f"ERRO: O conteúdo da chave 'verbetes_completo' em '{filepath}' não é uma lista.")
             return None # Retorna None se o conteúdo não for lista

        if not lista_verbetes:
             print(f"AVISO: A lista 'verbetes_completo' em '{filepath}' está vazia.")
             # Decide se retorna lista vazia ou None

        return lista_verbetes # Retorna a lista

    except json.JSONDecodeError as e:
        print(f"ERRO: Falha ao decodificar o JSON em '{filepath}': {e}")
        return None
    except Exception as e:
        print(f"Erro inesperado ao carregar JSON em {filepath}: {e}")
        return None

def construir_grafo_com_redirects(verbetes_com_flags: List[Dict[str, Any]],
                                  redirect_map: Dict[str, str],
                                  direcionado=True,
                                  max_redirect_hops=5) -> nx.Graph: # Adiciona limite de saltos
    """
    Constrói grafo resolvendo redirecionamentos, incluindo lógica para
    seguir cadeias de redirects (redirects de redirects).
    """
    print(f"Construindo grafo {'direcionado' if direcionado else 'não-direcionado'} com resolução ITERATIVA de redirects...")
    G = nx.DiGraph() if direcionado else nx.Graph()

    # --- 1. Filtrar verbetes reais e criar mapa titulo -> id ---
    verbetes_reais = []
    titulos_ids_reais = {}
    print("  - Filtrando verbetes reais e criando mapa Título->ID...")
    # Loop de filtragem e criação de titulos_ids_reais
    for index, verbete in enumerate(verbetes_com_flags):
         if not isinstance(verbete, dict): continue # Pula itens inválidos
         if not verbete.get('is_redirect') and 'id' in verbete and 'titulo' in verbete:
             verbetes_reais.append(verbete)
             titulos_ids_reais[verbete['titulo']] = str(verbete['id'])
    print(f"  - {len(verbetes_reais)} verbetes reais identificados.")
    if not verbetes_reais: return G

    # --- 2. Adicionar Nós (APENAS verbetes reais) ---
    print("  - Adicionando nós ao grafo...")
    # Loop de adição de nós
    for verbete in verbetes_reais: G.add_node(str(verbete['id']))
    print(f"  - {G.number_of_nodes()} nós adicionados.")

    # --- 3. Adicionar Arestas ---
    print("  - Adicionando arestas (resolvendo cadeias de redirecionamentos)...")
    arestas_adicionadas = 0
    refs_resolvidas_count = 0
    refs_quebradas_count = 0
    redirect_loops_detected = 0

    # TÍTULO EXATO DO VERBETE B (O PRIMEIRO REDIRECT)
    DEBUG_LINK_ALVO = "(Des)continuidades na experiência de \"vida sob cerco\" e na \"sociabilidade violenta\" (resenha)"
    # TÍTULO EXATO DO VERBETE A (A ORIGEM)
    DEBUG_VERBETE_ORIGEM = "Favela é comunidade? (artigo)"

    for verbete_origem in verbetes_reais:
        source_vid = str(verbete_origem['id'])
        referencias_originais = verbete_origem.get('referencias', [])
        
        # DEBUG: Flag para imprimir apenas sobre o verbete que nos interessa
        IS_DEBUG_VERBETE = (verbete_origem['titulo'] == DEBUG_VERBETE_ORIGEM)

        for ref_titulo in referencias_originais:
            if not isinstance(ref_titulo, str) or not ref_titulo: continue

            current_target_title = ref_titulo
            final_target_title = ref_titulo 
            visited_redirects = {ref_titulo} 
            hops = 0 
            
            # DEBUG: Verifica se é o link que estamos procurando
            IS_DEBUG_LINK = (ref_titulo == DEBUG_LINK_ALVO)
            if IS_DEBUG_VERBETE and IS_DEBUG_LINK:
                print("\n--- [DEBUG] INICIANDO RASTREAMENTO DA ARESTA PROBLEMA ---")
                print(f"  Origem: {verbete_origem['titulo']} (ID: {source_vid})")
                print(f"  Ref Título (B): '{ref_titulo}'")

            # Continua seguindo redirects ENQUANTO o alvo atual AINDA ESTIVER no mapa
            while current_target_title in redirect_map and hops < max_redirect_hops:
                next_target = redirect_map[current_target_title]
                
                if IS_DEBUG_VERBETE and IS_DEBUG_LINK:
                    print(f"  Salto {hops+1}: DE '{current_target_title}' -> PARA '{next_target}'")

                if next_target in visited_redirects:
                    redirect_loops_detected += 1
                    final_target_title = current_target_title 
                    break 

                final_target_title = next_target 
                current_target_title = next_target 
                visited_redirects.add(current_target_title)
                hops += 1
                if hops == 1: 
                    refs_resolvidas_count += 1
                        
            # --- PONTO CRÍTICO DA VERIFICAÇÃO ---
            if IS_DEBUG_VERBETE and IS_DEBUG_LINK:
                 print(f"  Cadeia resolvida. Alvo Final (D): '{final_target_title}'")
                 # Verificação crucial:
                 print(f"  Verificando se Alvo Final (D) está em 'titulos_ids_reais'...")
            
            # Verifica se o título FINAL (após seguir a cadeia) é um verbete real
            if final_target_title in titulos_ids_reais:
                target_vid = titulos_ids_reais[final_target_title]
                
                if IS_DEBUG_VERBETE and IS_DEBUG_LINK:
                    print(f"  [SUCESSO] Alvo Final (D) ENCONTRADO em 'titulos_ids_reais'. ID do Alvo: {target_vid}")
                    print(f"  Verificando se G.has_node({source_vid}) e G.has_node({target_vid})...")

                if G.has_node(source_vid) and G.has_node(target_vid):
                    G.add_edge(source_vid, target_vid)
                    arestas_adicionadas += 1
                    if IS_DEBUG_VERBETE and IS_DEBUG_LINK:
                        print(f"  [SUCESSO FINAL] Aresta {source_vid} -> {target_vid} ADICIONADA.")
                else:
                    if IS_DEBUG_VERBETE and IS_DEBUG_LINK:
                         print(f"  [FALHA 2] Um dos nós (origem ou destino) NÃO existe no grafo. Aresta IGNORADA.")
            else:
                refs_quebradas_count += 1
                if IS_DEBUG_VERBETE and IS_DEBUG_LINK:
                    print(f"  [FALHA 1] Alvo Final (D) '{final_target_title}' NÃO FOI ENCONTRADO em 'titulos_ids_reais'. Aresta IGNORADA.")
                    print("--- [DEBUG] FIM DO RASTREAMENTO ---")

    # --- 4. Conclusão e Retorno ---
    print(f"Construção do grafo concluída:")
    print(f"  - Nós (Verbetes Reais): {G.number_of_nodes()}")
    print(f"  - Arestas Adicionadas (tentativas): {arestas_adicionadas}")
    print(f"  - Arestas Únicas no Grafo: {G.number_of_edges()}")
    print(f"  - Referências que passaram por resolução (1+ saltos): {refs_resolvidas_count}")
    if redirect_loops_detected > 0:
         print(f"  - Cadeias de redirect interrompidas por loops: {redirect_loops_detected}")
    if refs_quebradas_count > 0:
        print(f"  - Referências ignoradas (alvo final inválido ou loop): {refs_quebradas_count}")
        
    return G

def salvar_dados_json(data_dict: Dict[str, Any], filepath: Path):
    print(f"Salvando dados em '{filepath}'...")
    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(data_dict, f, ensure_ascii=False, indent=2)
        print(f"✅ Arquivo salvo com sucesso em: '{filepath}'")
    except Exception as e:
        print(f"  [ERRO AO SALVAR JSON] em '{filepath}': {e}")



✅ Célula 1: Configurações e funções carregadas.


In [ ]:
# CÉLULA 2: CÁLCULO DE MÉTRICAS DE REDE
# Esta célula carrega os dados brutos, calcula todas as métricas de rede
# e salva um arquivo intermediário. Execute-a apenas quando os dados de entrada mudarem.

def calcular_metricas_grafo_limpo():
    """
    Carrega dados com flags e mapa de redirects, constrói grafo limpo,
    calcula métricas e salva os verbetes reais enriquecidos.
    """

    input_path = INPUT_DIR / INPUT_FILENAME_WITH_FLAGS
    redirect_map_path = INPUT_DIR / REDIRECT_MAP_FILENAME
    metrics_output_path = INPUT_DIR / METRICS_OUTPUT_FILENAME

    verbetes_com_flags = carregar_dados_json(input_path)
    if not verbetes_com_flags:
        print("ERRO: Falha ao carregar dados com flags.")
        return

    redirect_map = {}
    if redirect_map_path.exists():
        print(f"Carregando mapa de redirecionamentos de '{redirect_map_path}'...")
        with open(redirect_map_path, 'r', encoding='utf-8') as f:
            redirect_map = json.load(f)
        print(f"Mapa carregado com {len(redirect_map)} entradas.")
    else:
        print(f"AVISO: Arquivo de mapa de redirecionamentos '{redirect_map_path}' não encontrado. O grafo será construído sem resolver redirects.")

    # --- 3. Construir o Grafo Limpo ---
    # Chama a função que usa o redirect_map
    G = construir_grafo_com_redirects(verbetes_com_flags, redirect_map, direcionado=True)
    
    # Filtra a lista original para conter apenas os nós que REALMENTE estão no grafo G
    # (Ou seja, os verbetes que não eram redirects)
    verbetes_reais_no_grafo = [
        v for v in verbetes_com_flags
        if not v.get('is_redirect') and str(v.get('id')) in G # Garante que o nó existe no grafo final
    ]
    print(f"Total de {len(verbetes_reais_no_grafo)} verbetes reais (nós) serão enriquecidos.")


    # --- 4. Calcular Métricas ---
    print("\nCalculando métricas de rede complexas sobre o grafo limpo...")
    # Cria versão não direcionada para Clustering e Comunidades
    G_undirected = G.to_undirected()

    # --- 4.1 Detecção de Comunidades ---
    print(" - Detectando comunidades no componente gigante do grafo limpo...")
    partition = {} # Dicionário: node_id -> community_id
    community_colors = {} # Dicionário: community_id -> color_hex

    # Garante que o grafo não direcionado não está vazio
    if G_undirected.number_of_nodes() > 0:
        connected_components = list(nx.connected_components(G_undirected))
        if connected_components:
            # Encontra o componente gigante
            giant_component_nodes = max(connected_components, key=len)
            G_giant = G_undirected.subgraph(giant_component_nodes)
            print(f"   Componente gigante contém {G_giant.number_of_nodes()} nós.")

            # Executa Louvain no componente gigante
            if G_giant.number_of_nodes() > 0:
                 partition = community_louvain.best_partition(G_giant, resolution=LOUVAIN_RESOLUTION, random_state=RANDOM_SEED)
                 num_communities = len(set(partition.values()))
                 print(f"   Foram detectadas {num_communities} comunidades no componente gigante.")

                 # Gera cores distintas
                 random.seed(RANDOM_SEED)
                 cores_geradas = []
                 for i in range(num_communities):
                      hue = i / num_communities
                      saturation = 0.7 + random.random() * 0.2 # Evita cores muito pálidas
                      lightness = 0.5 + random.random() * 0.1 # Evita cores muito escuras/claras
                      rgb = colorsys.hls_to_rgb(hue, lightness, saturation)
                      hex_color = '#{:02x}{:02x}{:02x}'.format(int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))
                      cores_geradas.append(hex_color)
                 random.shuffle(cores_geradas) # Embaralha para evitar sequências parecidas
                 community_colors = {i: cores_geradas[i] for i in range(num_communities)}

            else:
                 print("   Componente gigante está vazio após subgrafo. Nenhuma comunidade detectada.")
        else:
             print("   Grafo não possui componentes conectados. Nenhuma comunidade detectada.")
    else:
        print("   Grafo não direcionado está vazio. Nenhuma métrica será calculada.")
        return


    # --- 4.2 Cálculo de Centralidades ---
    print(" - Calculando Centralidades (pode levar alguns minutos)...")
    # Calcula métricas apenas para os nós que existem em G
    nodes_in_G = list(G.nodes()) # Lista de IDs (strings) dos nós reais
    
    # Usar try-except para métricas que podem falhar em grafos desconectados
    try:
        betweenness = nx.betweenness_centrality(G, normalized=True, endpoints=False, k=None)
    except Exception as e:
        print(f"  [AVISO] Erro ao calcular Betweenness (pode ocorrer em grafos pequenos/desconectados): {e}. Usando 0.0.")
        betweenness = {node_id: 0.0 for node_id in nodes_in_G}
        
    try:
        closeness = nx.closeness_centrality(G)
    except Exception as e:
        print(f"  [AVISO] Erro ao calcular Closeness (pode ocorrer em grafos desconectados): {e}. Usando 0.0.")
        closeness = {node_id: 0.0 for node_id in nodes_in_G}

    pagerank = nx.pagerank(G, alpha=0.85)
    clustering = nx.clustering(G_undirected)

    # Coleta de graus
    in_degree = dict(G.in_degree())
    out_degree = dict(G.out_degree())
    total_degree = dict(G.degree())


    # --- 5. Enriquecimento dos Dados (Apenas Verbetes Reais) ---
    print("Enriquecendo o dataset (apenas verbetes reais) com as métricas...")
    verbetes_reais_enriquecidos = []
    
    for verbete in verbetes_reais_no_grafo: # Itera SÓ sobre os verbetes reais
        vid = str(verbete['id']) # ID já deve existir e ser string
        
        # Obtém o ID da comunidade (padrão -1 se não estiver no componente gigante)
        community_id = partition.get(vid, -1) 
        
        # Cria um novo dicionário ou atualiza o existente
        verbete_enriquecido = verbete.copy() # Copia os dados originais do verbete real
        
        verbete_enriquecido.update({
            'community_id': community_id,
            'community_color': community_colors.get(community_id, '#808080'), # Cinza para isolados
            'in_degree': in_degree.get(vid, 0),
            'out_degree': out_degree.get(vid, 0),
            'total_degree': total_degree.get(vid, 0),
            'betweenness_centrality': betweenness.get(vid, 0.0),
            'pagerank': pagerank.get(vid, 0.0),
            'clustering_coefficient': clustering.get(vid, 0.0),
            'closeness_centrality': closeness.get(vid, 0.0)
        })
        verbetes_reais_enriquecidos.append(verbete_enriquecido)

    # --- 6. Salvar o Resultado ---
    # Estrutura final para salvar contém APENAS os verbetes reais enriquecidos
    dados_finais_para_salvar = {
        'total_verbetes_reais': len(verbetes_reais_enriquecidos),
        'verbetes_completo': verbetes_reais_enriquecidos # Lista apenas com verbetes reais + métricas
    }
    
    salvar_dados_json(dados_finais_para_salvar, metrics_output_path)
    print(f"Verbetes reais enriquecidos com métricas salvos em '{metrics_output_path.resolve()}'")

# Chama a função principal desta célula
calcular_metricas_grafo_limpo()


Carregando dados de '..\dados\dados_com_flags_redirecionamento\dados_api_com_flags_e_refs.json'...
Carregando mapa de redirecionamentos de '..\dados\dados_com_flags_redirecionamento\redirect_map.json'...
Mapa carregado com 748 entradas.
Construindo grafo direcionado com resolução ITERATIVA de redirects...
  - Filtrando verbetes reais e criando mapa Título->ID...
  - 3294 verbetes reais identificados.
  - Adicionando nós ao grafo...
  - 3294 nós adicionados.
  - Adicionando arestas (resolvendo cadeias de redirecionamentos)...

--- [DEBUG] INICIANDO RASTREAMENTO DA ARESTA PROBLEMA ---
  Origem: Favela é comunidade? (artigo) (ID: 1607)
  Ref Título (B): '(Des)continuidades na experiência de "vida sob cerco" e na "sociabilidade violenta" (resenha)'
  Salto 1: DE '(Des)continuidades na experiência de "vida sob cerco" e na "sociabilidade violenta" (resenha)' -> PARA 'Descontinuidades na experiência de "vida sob cerco" e na "sociabilidade violenta" (resenha)'
  Salto 2: DE 'Descontinuidades n

In [ ]:
# CÉLULA 3: CÁLCULO DE LAYOUT INICIAL (KAMADA-KAWAI + GRELHA)

FORCAR_RECALCULO_DISTANCIAS = True 

def calcular_e_salvar_layout_inicial():
    """
    Carrega dados, reconstrói grafo NÃO-DIRECIONADO usando a MESMA lógica
    da Célula 2 (resolvendo redirects e normalizando), recalcula distâncias e calcula o layout.
    """
    
    # --- 1. Definir Caminhos ---
    metrics_path = INPUT_DIR / METRICS_OUTPUT_FILENAME
    positions_output_path = INPUT_DIR / POSITIONS_OUTPUT_FILENAME
    shortest_paths_path = INPUT_DIR / SHORTEST_PATHS_FILENAME
    redirect_map_path = INPUT_DIR / REDIRECT_MAP_FILENAME
    
    print("-" * 30)
    print("Iniciando Célula 3: Cálculo de Layout (Versão Corrigida)")
    print("-" * 30)
    
    # --- 2. Carregar Dados Enriquecidos (Saída da Célula 2) ---
    print(f"Carregando dados enriquecidos de '{metrics_path}'...")
    verbetes = carregar_dados_json(metrics_path) # Contém APENAS verbetes reais
    if not verbetes:
         print("ERRO CRÍTICO: Falha ao carregar verbetes enriquecidos da Célula 2. Abortando.")
         return 

    # --- Carregar o redirect_map (necessário para reconstruir o grafo) ---
    redirect_map = {}
    if redirect_map_path.exists():
        print(f"Carregando mapa de redirecionamentos de '{redirect_map_path}'...")
        try:
            with open(redirect_map_path, 'r', encoding='utf-8') as f:
                redirect_map = json.load(f)
            print(f"Mapa carregado com {len(redirect_map)} entradas.")
        except Exception as e:
             print(f"ERRO ao carregar redirect_map: {e}. Grafo será construído sem ele.")
    else:
        print(f"AVISO: Arquivo de mapa de redirecionamentos '{redirect_map_path}' não encontrado.")


    # --- 3. Reconstruir Grafo Não-Direcionado  ---
    print("Reconstruindo grafo não-direcionado (usando lógica da Célula 2)...")
    try:
        # Carrega a lista COMPLETA com flags (necessária para construir_grafo_com_redirects)
        input_flags_path = INPUT_DIR / INPUT_FILENAME_WITH_FLAGS
        print(f"Recarregando lista completa com flags de '{input_flags_path}'...")
        verbetes_com_flags = carregar_dados_json(input_flags_path)
        if not verbetes_com_flags:
             print("ERRO CRÍTICO: Falha ao carregar 'dados_api_com_flags_e_refs.json'. Abortando.")
             return

        G_undirected = construir_grafo_com_redirects(
            verbetes_com_flags, 
            redirect_map, 
            direcionado=False 
        )
    except Exception as e:
        print(f"ERRO CRÍTICO ao reconstruir grafo não-direcionado: {e}. Abortando.")
        return

    if G_undirected.number_of_nodes() == 0: 
         print("ERRO CRÍTICO: Grafo não-direcionado reconstruído está vazio. Abortando.")
         return
         
    # --- 4. Separar Componente Gigante e Isolados ---
    print("\nIdentificando componente gigante e nós isolados...")

    giant_component_nodes = set()
    G_giant = nx.Graph()
    isolated_nodes = []
    try:
        if G_undirected.number_of_nodes() > 0:
            connected_components = list(nx.connected_components(G_undirected))
            if connected_components:
                giant_component_nodes = max(connected_components, key=len)
                G_giant = G_undirected.subgraph(giant_component_nodes).copy()
                isolated_nodes = [node for node in G_undirected.nodes() if node not in giant_component_nodes]
            else:
                 isolated_nodes = list(G_undirected.nodes())
        else:
             isolated_nodes = []
        print(f"  - Componente gigante: {G_giant.number_of_nodes()} nós.")
        print(f"  - Nós isolados: {len(isolated_nodes)} nós.")
    except Exception as e:
        print(f"ERRO ao separar componentes: {e}."); giant_component_nodes=set(); G_giant=nx.Graph(); isolated_nodes=list(G_undirected.nodes())

    # --- 5. Calcular/Carregar Distâncias ---
    shortest_path_dist = None
    print("\n--- Verificação/Cálculo de Distâncias ---")
    if FORCAR_RECALCULO_DISTANCIAS and shortest_paths_path.exists():
        print(f"FORÇANDO RECÁLCULO: Deletando cache antigo '{shortest_paths_path.name}'...")
        try:
            shortest_paths_path.unlink()
            print("  - Cache antigo deletado.")
        except OSError as e:
            print(f"  [ERRO] Não foi possível deletar o cache: {e}.")
    
    if not FORCAR_RECALCULO_DISTANCIAS and shortest_paths_path.exists():
        try:
            print(f"Carregando distâncias pré-calculadas de '{shortest_paths_path}'...")
            with open(shortest_paths_path, 'rb') as f: cached_data = pickle.load(f)
            if isinstance(cached_data, dict) and len(cached_data) == G_giant.number_of_nodes():
                 shortest_path_dist = cached_data
                 print("  Cache carregado com sucesso.")
            else: print("  [AVISO] Cache inconsistente. Recalculando...")
        except Exception as e: print(f"  [AVISO] Erro ao carregar cache: {e}. Recalculando...")

    if shortest_path_dist is None and G_giant.number_of_nodes() > 1:
        print("Calculando distâncias de caminho mais curto (pode demorar)...")
        start_time = time.time()
        try:
             path_length_generator = nx.all_pairs_shortest_path_length(G_giant)
             shortest_path_dist = {source: targets for source, targets in path_length_generator}
             end_time = time.time(); print(f"  Distâncias calculadas em {end_time - start_time:.2f} segundos.")
             print(f"Salvando novas distâncias calculadas em '{shortest_paths_path}'...")
             with open(shortest_paths_path, 'wb') as f: pickle.dump(shortest_path_dist, f, pickle.HIGHEST_PROTOCOL)
             print("  Distâncias salvas.")
        except Exception as e_calc: print(f"  [ERRO] Falha ao calcular/salvar distâncias: {e_calc}."); shortest_path_dist = None
    elif G_giant.number_of_nodes() <= 1: print("Componente gigante muito pequeno. Pulando cálculo de distâncias.")

    # --- 6. Calcular Layout Kamada-Kawai ---
    posicoes_finais = {} 
    if G_giant.number_of_nodes() > 0:
         print(f"\nCalculando layout Kamada-Kawai para {G_giant.number_of_nodes()} nós...")
         start_time = time.time()
         try:
            posicoes_relativas = nx.kamada_kawai_layout(G_giant, dist=shortest_path_dist, scale=LAYOUT_SCALE_FACTOR)
            posicoes_finais = {node_id: {'x': float(pos[0]), 'y': float(pos[1])} for node_id, pos in posicoes_relativas.items()}
            end_time = time.time(); print(f"  Layout Kamada-Kawai calculado em {end_time - start_time:.2f} segundos.")
         except Exception as e_kk:
              print(f"  [ERRO] Falha ao calcular Kamada-Kawai: {e_kk}")
              print("  Tentando layout spring como fallback...")
              start_time_spring = time.time()
              try:
                   posicoes_relativas = nx.spring_layout(G_giant, k=None, iterations=50, seed=RANDOM_SEED, scale=LAYOUT_SCALE_FACTOR)
                   posicoes_finais = {nid: {'x': float(p[0]), 'y': float(p[1])} for nid, p in posicoes_relativas.items()}
                   end_time_spring = time.time(); print(f"  Layout Spring (fallback) calculado em {end_time_spring - start_time_spring:.2f} segundos.")
              except Exception as e_spring:
                   print(f"  [ERRO CRÍTICO] Fallback para spring_layout também falhou: {e_spring}")
                   posicoes_finais = {node_id: {'x': 0.0, 'y': 0.0} for node_id in G_giant.nodes()}
    else:
         print("Componente gigante está vazio. Pulando cálculo de layout principal.")

    # --- 7. Posicionar Nós Isolados em Grelha ---
    print(f"\nPosicionando {len(isolated_nodes)} nós isolados em grelha...")
    if isolated_nodes:
        try:
            max_x_main_graph = max(pos['x'] for pos in posicoes_finais.values()) if posicoes_finais else 0
            grid_start_x = max_x_main_graph + (LAYOUT_SCALE_FACTOR if LAYOUT_SCALE_FACTOR else 2000) 
            spacing = max(100, LAYOUT_SCALE_FACTOR / 15 if LAYOUT_SCALE_FACTOR else 150) 
            cols = int(math.sqrt(len(isolated_nodes))) + 1
            start_y = 0 
            for i, node_id in enumerate(isolated_nodes):
                row = i // cols
                col = i % cols
                posicoes_finais[node_id] = {'x': float(grid_start_x + (col * spacing)), 'y': float(start_y + (row * spacing))}
            print(f"  {len(isolated_nodes)} nós isolados posicionados.")
        except Exception as e_grid:
             print(f"  [ERRO] Falha ao posicionar nós isolados: {e_grid}")
             for node_id in isolated_nodes:
                  if node_id not in posicoes_finais: posicoes_finais[node_id] = {'x': 0.0, 'y': 0.0}

    # --- 8. Adicionar Posições ao Dataset ---
    print("\nAdicionando posições calculadas ao dataset de verbetes...")

    for verbete in verbetes: 
        node_id_str = str(verbete.get('id'))
        position = posicoes_finais.get(node_id_str, {'x': 0.0, 'y': 0.0}) 
        verbete['position'] = position
            
    # --- 9. Salvar Resultado Final ---
    print(f"\nPreparando dados para salvar em '{positions_output_path}'...")
    dados_para_salvar_layout = {
        'total_verbetes_reais': len(verbetes),
        'verbetes_completo': verbetes # Salva a lista de verbetes reais + métricas + posições
    }
    
    salvar_dados_json(dados_para_salvar_layout, positions_output_path)
    
    print("-" * 30)
    print("Célula 3: Cálculo de Layout Concluído")
    print("-" * 30)

# --- Execução da Célula ---
calcular_e_salvar_layout_inicial()

------------------------------
Iniciando Célula 3: Cálculo de Layout (Versão Corrigida)
------------------------------
Carregando dados enriquecidos de '..\dados\dados_com_flags_redirecionamento\dados_com_metricas_v2_novo.json'...
Carregando dados de '..\dados\dados_com_flags_redirecionamento\dados_com_metricas_v2_novo.json'...
Carregando mapa de redirecionamentos de '..\dados\dados_com_flags_redirecionamento\redirect_map.json'...
Mapa carregado com 748 entradas.
Reconstruindo grafo não-direcionado (usando lógica da Célula 2)...
Recarregando lista completa com flags de '..\dados\dados_com_flags_redirecionamento\dados_api_com_flags_e_refs.json'...
Carregando dados de '..\dados\dados_com_flags_redirecionamento\dados_api_com_flags_e_refs.json'...
Construindo grafo não-direcionado com resolução ITERATIVA de redirects...
  - Filtrando verbetes reais e criando mapa Título->ID...
  - 3294 verbetes reais identificados.
  - Adicionando nós ao grafo...
  - 3294 nós adicionados.
  - Adicionando a

In [ ]:
# CÉLULA 4: REFINAMENTO DO LAYOUT PARA EVITAR SOBREPOSIÇÃO
# Esta célula é ideal para ser executada várias vezes, ajustando os parâmetros
# do spring_layout para encontrar a melhor visualização.

def refinar_e_salvar_layout_final():
    """
    Carrega dados com posições (da Célula 3), recarrega dados brutos (do Passo 1 e 2),
    reconstrói o grafo EXATAMENTE como na Célula 2, e aplica o refinamento
    de layout (spring_layout) para salvar o resultado final.
    """
    
    # --- 1. Definir Caminhos ---
    # Entrada com Posições (Saída da Célula 3)
    positions_path = INPUT_DIR / POSITIONS_OUTPUT_FILENAME 
    # Saída Final desta Célula
    final_output_path = INPUT_DIR / FINAL_REFINED_OUTPUT_FILENAME 

    # Entradas necessárias para RECONSTRUIR o grafo
    input_flags_path = INPUT_DIR / INPUT_FILENAME_WITH_FLAGS
    redirect_map_path = INPUT_DIR / REDIRECT_MAP_FILENAME
    
    print("-" * 30)
    print("Iniciando Célula 4: Refinamento do Layout (Versão Corrigida)")
    print("-" * 30)

    # --- 2. Carregar Dados com Posições Iniciais ---
    print(f"Carregando dados com posições de '{positions_path}'...")
    verbetes = carregar_dados_json(positions_path) # Lista SÓ com verbetes reais + métricas + posições
    if not verbetes:
        print("ERRO CRÍTICO: Falha ao carregar dados da Célula 3. Abortando.")
        return

    # --- 3. Carregar Dados Brutos para Reconstrução do Grafo ---
    print(f"Recarregando lista completa com flags de '{input_flags_path}'...")
    verbetes_com_flags = carregar_dados_json(input_flags_path)
    if not verbetes_com_flags:
         print("ERRO CRÍTICO: Falha ao carregar 'dados_api_com_flags_e_refs.json'. Abortando.")
         return

    print(f"Recarregando mapa de redirecionamentos de '{redirect_map_path}'...")
    redirect_map = {}
    if redirect_map_path.exists():
        try:
            with open(redirect_map_path, 'r', encoding='utf-8') as f:
                redirect_map = json.load(f)
            print(f"Mapa carregado com {len(redirect_map)} entradas.")
        except Exception as e:
             print(f"ERRO ao carregar redirect_map: {e}. Grafo será construído sem ele.")
    else:
        print(f"AVISO: Arquivo de mapa de redirecionamentos '{redirect_map_path}' não encontrado.")

    # --- 4. Reconstruir Grafo Não-Direcionado ---
    print("Reconstruindo grafo não-direcionado (usando lógica da Célula 2)...")
    try:

        G_undirected = construir_grafo_com_redirects(
            verbetes_com_flags, 
            redirect_map, 
            direcionado=False
        )
    except Exception as e:
        print(f"ERRO CRÍTICO ao reconstruir grafo não-direcionado: {e}. Abortando.")
        return

    if G_undirected.number_of_nodes() == 0:
         print("ERRO CRÍTICO: Grafo reconstruído está vazio. Verifique os dados de entrada. Abortando.")
         return
         
    # --- 5. Extrair Posições Iniciais (dos 'verbetes' carregados) e Normalizar ---
    print("\nExtraindo e normalizando posições iniciais (da Célula 3)...")
    posicoes_iniciais_scaled = {}
    for v in verbetes:
        node_id_str = str(v.get('id'))
        pos = v.get('position')
        if isinstance(pos, dict) and 'x' in pos and 'y' in pos:
             posicoes_iniciais_scaled[node_id_str] = (pos['x'], pos['y'])
        else:
             print(f"  [AVISO] Posição inicial ausente/inválida para nó ID {node_id_str}. Usando (0,0).")
             posicoes_iniciais_scaled[node_id_str] = (0.0, 0.0)

    max_abs_coord = max(max(abs(x), abs(y)) for x, y in posicoes_iniciais_scaled.values()) if posicoes_iniciais_scaled else 1.0
    if max_abs_coord == 0: max_abs_coord = 1.0 
    
    posicoes_iniciais_unscaled = {
        node: (x / max_abs_coord, y / max_abs_coord) 
        for node, (x, y) in posicoes_iniciais_scaled.items()
    }

    # --- 6. Identificar Componente Gigante ---
    print("Identificando componente gigante para refinamento...")
    giant_component_nodes = set()
    G_giant = nx.Graph()
    try:
        if G_undirected.number_of_nodes() > 0:
            connected_components = list(nx.connected_components(G_undirected))
            if connected_components:
                giant_component_nodes = max(connected_components, key=len)
                G_giant = G_undirected.subgraph(giant_component_nodes).copy()
        print(f"  - Componente gigante para refinamento: {G_giant.number_of_nodes()} nós.")
    except Exception as e:
        print(f"ERRO ao identificar componente gigante: {e}")
        G_giant = nx.Graph()

    # --- 7. Refinar Layout do Componente Gigante com Spring Layout ---
    posicoes_finais_refinadas = {} 
    if G_giant.number_of_nodes() > 0:
        pos_giant_iniciais_unscaled = {
            node: pos for node, pos in posicoes_iniciais_unscaled.items() if node in giant_component_nodes
        }
        
        if len(pos_giant_iniciais_unscaled) != G_giant.number_of_nodes():
             print("  [AVISO] Inconsistência: Nem todos os nós do gigante tinham posições. Spring layout pode começar do zero.")
             pos_giant_iniciais_unscaled = None 

        print(f"Refinando layout de {G_giant.number_of_nodes()} nós do componente gigante usando spring_layout...")
        start_time = time.time()
        try:
            k_value = 0.4 / math.sqrt(G_giant.number_of_nodes()) if G_giant.number_of_nodes() > 0 else None
            
            pos_ajustadas_giant_unscaled = nx.spring_layout(
                G_giant, 
                pos=pos_giant_iniciais_unscaled, 
                fixed=None, 
                iterations=80, 
                seed=RANDOM_SEED,
                k=k_value, 
                scale=1.0, 
                center=None
            )
            end_time = time.time()
            print(f"  Refinamento concluído em {end_time - start_time:.2f} segundos.")

            for node_id, pos_unscaled in pos_ajustadas_giant_unscaled.items():
                 posicoes_finais_refinadas[node_id] = {
                     'x': float(pos_unscaled[0] * max_abs_coord), 
                     'y': float(pos_unscaled[1] * max_abs_coord)
                 }
        except Exception as e_spring:
            print(f"  [ERRO] Falha ao refinar com spring_layout: {e_spring}")
            print("  Usando as posições originais (não refinadas) para o componente gigante.")
            for node_id in giant_component_nodes:
                 if node_id in posicoes_iniciais_scaled:
                      posicoes_finais_refinadas[node_id] = {
                          'x': float(posicoes_iniciais_scaled[node_id][0]),
                          'y': float(posicoes_iniciais_scaled[node_id][1])
                      }
                 else:
                      posicoes_finais_refinadas[node_id] = {'x': 0.0, 'y': 0.0}
    else:
        print("Componente gigante vazio. Pulando refinamento.")

    # --- 8. Combinar Posições Refinadas e Isoladas ---
    print("\nCombinando posições refinadas (gigante) e originais (isolados)...")
    isolados_count = 0
    for node_id_str, pos_scaled in posicoes_iniciais_scaled.items():
        if node_id_str not in posicoes_finais_refinadas:
             posicoes_finais_refinadas[node_id_str] = {'x': float(pos_scaled[0]), 'y': float(pos_scaled[1])}
             isolados_count += 1
    print(f"  Posições de {isolados_count} nós isolados (grelha) mantidas.")

    # --- 9. Atualizar Dataset Final ---
    print("Atualizando o dataset final com as posições refinadas...")
    for verbete in verbetes:
        verbete_id_str = str(verbete.get('id'))
        final_position = posicoes_finais_refinadas.get(verbete_id_str)
        if final_position:
            verbete['position'] = final_position
        else:
            print(f"  [AVISO] Posição final não encontrada para nó ID {verbete_id_str}.")
            if 'position' not in verbete: 
                 verbete['position'] = {'x': 0.0, 'y': 0.0}

    # --- 10. Salvar Resultado Final ---
    print(f"\nPreparando dados para salvar em '{final_output_path}'...")
    dados_para_salvar_final = {
        'total_verbetes_reais': len(verbetes),
        'verbetes_completo': verbetes # Salva a lista 'verbetes' atualizada com 'position'
    }
    
    salvar_dados_json(dados_para_salvar_final, final_output_path)
    
    print("-" * 30)
    print("Célula 4: Refinamento de Layout Concluído")
    print("-" * 30)

# --- Execução da Célula ---
refinar_e_salvar_layout_final()


------------------------------
Iniciando Célula 4: Refinamento do Layout (Versão Corrigida)
------------------------------
Carregando dados com posições de '..\dados\dados_com_flags_redirecionamento\dados_com_posicoes_v2_novo.json'...
Carregando dados de '..\dados\dados_com_flags_redirecionamento\dados_com_posicoes_v2_novo.json'...
Recarregando lista completa com flags de '..\dados\dados_com_flags_redirecionamento\dados_api_com_flags_e_refs.json'...
Carregando dados de '..\dados\dados_com_flags_redirecionamento\dados_api_com_flags_e_refs.json'...
Recarregando mapa de redirecionamentos de '..\dados\dados_com_flags_redirecionamento\redirect_map.json'...
Mapa carregado com 748 entradas.
Reconstruindo grafo não-direcionado (usando lógica da Célula 2)...
Construindo grafo não-direcionado com resolução ITERATIVA de redirects...
  - Filtrando verbetes reais e criando mapa Título->ID...
  - 3294 verbetes reais identificados.
  - Adicionando nós ao grafo...
  - 3294 nós adicionados.
  - Adicion

In [8]:
# calcular_metrica_composta.py
#
# Este script lê os dados finais da rede e calcula uma métrica composta
# para identificar os "super-nós" — verbetes que são importantes em múltiplas dimensões.
#
# Metodologia Estatística:
# 1. Normalização Min-Max: Cada uma das cinco métricas selecionadas é normalizada
#    para uma escala de 0 a 1. Isso garante que cada métrica contribua de forma
#    equitativa para o resultado final, independentemente da sua escala original.
#    Fórmula: valor_normalizado = (valor - min) / (max - min)
# 2. Soma Ponderada (com pesos iguais): A métrica composta é a soma das
#    métricas normalizadas. Esta abordagem é mais robusta que a multiplicação,
#    pois não anula a pontuação de um nó que seja fraco em uma única dimensão.

def calcular_metrica_composta():
    """
    Calcula uma métrica de importância composta normalizando e somando
    diferentes indicadores de centralidade e atividade.
    """
    # --- 1. CONFIGURAÇÃO DE ARQUIVOS ---
    OUTPUT_FILENAME = 'dados_com_metrica_composta_novo.json'
    
    input_path = INPUT_DIR / FINAL_REFINED_OUTPUT_FILENAME
    output_path = INPUT_DIR / OUTPUT_FILENAME

    if not input_path.exists():
        print(f"ERRO: O arquivo de entrada não foi encontrado em '{input_path}'")
        return

    # --- 2. CARREGAMENTO E PREPARAÇÃO DOS DADOS ---
    print(f"Carregando dados de '{input_path}'...")
    with open(input_path, 'r', encoding='utf-8') as f:
        dados = json.load(f)
    
    verbetes = dados.get('verbetes_completo', [])
    if not verbetes:
        print("Nenhum verbete encontrado no arquivo.")
        return
        
    # Usar o Pandas facilita muito a normalização de colunas
    df = pd.DataFrame(verbetes)
    
    # Lista das métricas que farão parte do índice composto
    metricas_para_normalizar = [
        'betweenness_centrality',
        'closeness_centrality',
        'total_degree',
        'quantidade_edicoes',
        'pagerank'
    ]

    print("\nNormalizando as métricas para uma escala de 0 a 1...")
    for metrica in metricas_para_normalizar:
        if metrica in df.columns:
            min_val = df[metrica].min()
            max_val = df[metrica].max()
            
            # Evita divisão por zero se todos os valores forem iguais
            if (max_val - min_val) > 0:
                df[f'{metrica}_norm'] = (df[metrica] - min_val) / (max_val - min_val)
            else:
                df[f'{metrica}_norm'] = 0 # Se não há variação, a contribuição é nula
            
            print(f" - Métrica '{metrica}' normalizada.")
        else:
            print(f" - AVISO: Métrica '{metrica}' não encontrada no dataset.")
            df[f'{metrica}_norm'] = 0 # Adiciona uma coluna de zeros se a métrica não existir

    # --- 3. CÁLCULO DA MÉTRICA COMPOSTA ---
    print("\nCalculando a métrica composta pela soma dos valores normalizados...")
    
    # A métrica composta é a soma das versões normalizadas de cada indicador
    df['metrica_composta'] = (
        df['betweenness_centrality_norm'] +
        df['closeness_centrality_norm'] +
        df['total_degree_norm'] +
        df['quantidade_edicoes_norm'] +
        df['pagerank_norm']
    )
    
    print("Métrica composta calculada com sucesso.")

    # --- 4. ANÁLISE DOS RESULTADOS ---
    print("\n--- TOP 20 VERBETES POR MÉTRICA COMPOSTA ---")
    
    # Ordena o DataFrame pela nova métrica para encontrar os nós mais importantes
    df_sorted = df.sort_values(by='metrica_composta', ascending=False)
    
    for index, row in df_sorted.head(20).iterrows():
        print(f"  - Título: {row['titulo']:<70} | Pontuação Composta: {row['metrica_composta']:.4f}")

    # --- 5. SALVANDO O ARQUIVO FINAL ---
    # Converte o DataFrame de volta para o formato de dicionário original
    verbetes_finais = df_sorted.to_dict(orient='records')
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump({'verbetes_completo': verbetes_finais}, f, ensure_ascii=False, indent=2)
        
    print(f"\n✅ Arquivo com a métrica composta salvo em '{output_path}'!")


if __name__ == '__main__':
    calcular_metrica_composta()


Carregando dados de '..\dados\dados_com_flags_redirecionamento\dados_com_posicoes_v3_novo.json'...

Normalizando as métricas para uma escala de 0 a 1...
 - Métrica 'betweenness_centrality' normalizada.
 - Métrica 'closeness_centrality' normalizada.
 - Métrica 'total_degree' normalizada.
 - Métrica 'quantidade_edicoes' normalizada.
 - Métrica 'pagerank' normalizada.

Calculando a métrica composta pela soma dos valores normalizados...
Métrica composta calculada com sucesso.

--- TOP 20 VERBETES POR MÉTRICA COMPOSTA ---
  - Título: Lista de Favelas do Município do Rio de Janeiro                        | Pontuação Composta: 4.2268
  - Título: Complexo da Maré                                                       | Pontuação Composta: 2.7507
  - Título: Dicionário de Favelas Marielle Franco                                  | Pontuação Composta: 2.4071
  - Título: Marielle Franco                                                        | Pontuação Composta: 2.0714
  - Título: Redes da Maré    

In [9]:
# calcular_constraint.py
#
# Este script lê os dados enriquecidos da rede, reconstrói o grafo
# e calcula a métrica de Restrição (Constraint) de Burt para cada nó.
# A métrica é então adicionada ao dataset, que é salvo em um novo arquivo.


def calcular_constraint_para_nos():
    """
    Carrega dados (com métrica composta), reconstrói o grafo DIRECIONADO
    (usando a lógica correta da Célula 2), calcula Constraint e salva.
    """
    
    # --- 1. CONFIGURAÇÃO DE ARQUIVOS ---    
    INPUT_FILENAME = 'dados_com_metrica_composta_novo.json'
    OUTPUT_FILENAME = 'dados_com_constraint_novo.json'

    input_path = INPUT_DIR / INPUT_FILENAME
    output_path = INPUT_DIR / OUTPUT_FILENAME
    
    # Caminhos para os dados BRUTOS necessários para RECONSTRUIR o grafo
    input_flags_path = INPUT_DIR / INPUT_FILENAME_WITH_FLAGS
    redirect_map_path = INPUT_DIR / REDIRECT_MAP_FILENAME

    print("-" * 30)
    print("Iniciando Célula 6: Cálculo de Constraint (Versão Corrigida)")
    print("-" * 30)

    # --- 2. Carregar Dados a Serem Atualizados (da Célula 5) ---
    if not input_path.exists():
        print(f"ERRO CRÍTICO: O arquivo de entrada da Célula 5 não foi encontrado em '{input_path}'. Abortando.")
        return
    print(f"Carregando dados (com métrica composta) de '{input_path}'...")
    verbetes_para_atualizar = carregar_dados_json(input_path) 
    if not verbetes_para_atualizar:
        print("ERRO CRÍTICO: Falha ao carregar dados da Célula 5. Abortando.")
        return

    # --- 3. Carregar Dados Brutos para Reconstrução do Grafo ---
    print(f"Recarregando lista completa com flags de '{input_flags_path}'...")
    verbetes_com_flags = carregar_dados_json(input_flags_path)
    if not verbetes_com_flags:
         print("ERRO CRÍTICO: Falha ao carregar 'dados_api_com_flags_e_refs.json'. Abortando.")
         return

    print(f"Recarregando mapa de redirecionamentos de '{redirect_map_path}'...")
    redirect_map = {}
    if redirect_map_path.exists():
        try:
            with open(redirect_map_path, 'r', encoding='utf-8') as f:
                redirect_map = json.load(f)
            print(f"Mapa carregado com {len(redirect_map)} entradas.")
        except Exception as e:
             print(f"ERRO ao carregar redirect_map: {e}. Grafo será construído sem ele.")
    else:
        print(f"AVISO: Arquivo de mapa de redirecionamentos '{redirect_map_path}' não encontrado.")

    # --- 4. Reconstruir Grafo Direcionado (LÓGICA CORRETA) ---
    print("Reconstruindo grafo direcionado (usando lógica da Célula 2)...")
    try:
        G = construir_grafo_com_redirects(
            verbetes_com_flags, 
            redirect_map, 
            direcionado=True
        )
    except Exception as e:
        print(f"ERRO CRÍTICO ao reconstruir grafo direcionado: {e}. Abortando.")
        return

    if G.number_of_nodes() == 0:
         print("ERRO CRÍTICO: Grafo reconstruído está vazio. Verifique os dados de entrada. Abortando.")
         return

    # --- 5. CÁLCULO DA MÉTRICA DE CONSTRAINT ---
    print("\nCalculando a métrica de Restrição (Constraint) para cada nó...")
    start_time = time.time()
    try:
        # Calcula a métrica no grafo direcionado (consistente com Célula 2)
        constraint_scores = nx.constraint(G)
        end_time = time.time()
        print(f"  Cálculo da métrica de Constraint concluído em {end_time - start_time:.2f} segundos.")
    except Exception as e:
        print(f"  [ERRO] Falha ao calcular Constraint: {e}. A métrica não será adicionada.")
        constraint_scores = {}

    # --- 6. ENRIQUECIMENTO DOS DADOS ---
    print("Adicionando a métrica de Constraint ao conjunto de dados...")
    nodes_atualizados = 0
    # Itera sobre a lista carregada da Célula 5
    for verbete in verbetes_para_atualizar: 
        verbete_id = str(verbete.get('id'))
        # Adiciona o valor da constraint
        verbete['constraint'] = constraint_scores.get(verbete_id, None)
        nodes_atualizados += 1
    print(f"  Métrica de Constraint adicionada/atualizada para {nodes_atualizados} verbetes.")

    # --- 7. SALVANDO O ARQUIVO FINAL ---
    print(f"\nSalvando dados finais (com Constraint) em '{output_path}'...")
    dados_para_salvar_final = {
         'total_verbetes_reais': len(verbetes_para_atualizar),
         'verbetes_completo': verbetes_para_atualizar # Salva a lista já com todas as métricas
    }
    salvar_dados_json(dados_para_salvar_final, output_path)

    print("-" * 30)
    print("Célula 6: Cálculo de Constraint Concluído")
    print("-" * 30)

# --- Execução da Célula ---
calcular_constraint_para_nos()


------------------------------
Iniciando Célula 6: Cálculo de Constraint (Versão Corrigida)
------------------------------
Carregando dados (com métrica composta) de '..\dados\dados_com_flags_redirecionamento\dados_com_metrica_composta_novo.json'...
Carregando dados de '..\dados\dados_com_flags_redirecionamento\dados_com_metrica_composta_novo.json'...
Recarregando lista completa com flags de '..\dados\dados_com_flags_redirecionamento\dados_api_com_flags_e_refs.json'...
Carregando dados de '..\dados\dados_com_flags_redirecionamento\dados_api_com_flags_e_refs.json'...
Recarregando mapa de redirecionamentos de '..\dados\dados_com_flags_redirecionamento\redirect_map.json'...
Mapa carregado com 748 entradas.
Reconstruindo grafo direcionado (usando lógica da Célula 2)...
Construindo grafo direcionado com resolução ITERATIVA de redirects...
  - Filtrando verbetes reais e criando mapa Título->ID...
  - 3294 verbetes reais identificados.
  - Adicionando nós ao grafo...
  - 3294 nós adicionados.